In [1]:
from mainnet_launch.constants import *
from mainnet_launch.data_fetching.get_events import *
from mainnet_launch.abis import *

contract = ETH_CHAIN.client.eth.contract(AUTO_USD.autopool_addr, abi=AUTOPOOL_VAULT_NO_STRATEGY_CONTRACT_ABI)

In [2]:
events = get_each_event_in_contract(contract, AUTO_USD.chain, start_block=19_000_000, end_block=ETH_CHAIN.client.eth.block_number)

for k, v in events.items():
    print(k, v.shape)

AddedToRemovalQueue (2, 6)
Approval (62, 8)
Deposit (33, 9)
DestinationDebtReporting (8, 9)
DestinationVaultAdded (37, 6)
DestinationVaultRemoved (19, 6)
FeeCollected (0, 10)
FeeSinkSet (0, 6)
Initialized (1, 6)
LastPeriodicFeeTakeSet (5, 6)
Nav (63, 8)
NewNavShareFeeMark (8, 7)
NewProfitUnlockTime (1, 6)
NewTotalAssetsHighWatermark (5, 7)
Paused (0, 6)
PeriodicFeeCollected (0, 8)
PeriodicFeeSet (0, 6)
PeriodicFeeSinkSet (0, 6)
RebalanceCompleted (2, 6)
RebalanceFeeHighWaterMarkEnabledSet (0, 6)
RebalanceStarted (2, 7)
RemovedFromRemovalQueue (2, 6)
RewarderSet (1, 7)
Shutdown (0, 6)
StreamingFeeSet (0, 6)
SymbolAndDescSet (0, 7)
TokensRecovered (0, 8)
Transfer (84, 8)
Unpaused (0, 6)
Withdraw (22, 10)
WithdrawalQueueSet (0, 6)


In [3]:
import pandas as pd


RebalanceCompleted_df = events['RebalanceCompleted']

def _handle_rebalance_completed(RebalanceCompleted_df:pd.DataFrame):
    def _asset_changes_to_dict(asset_changes_tuple):
        return {
            "startingIdle": asset_changes_tuple[0],
            "startingDebt": asset_changes_tuple[1],
            "startingTotalSupply": asset_changes_tuple[2],
            "newIdle": asset_changes_tuple[3],
            "newDebt": asset_changes_tuple[4],
            "endingTotalSupply": asset_changes_tuple[5]
        }
    

    RebalanceCompleted_df[['startingIdle','startingDebt', 'startingTotalSupply','newIdle', 'newDebt', 'endingTotalSupply']] = pd.DataFrame.from_records(RebalanceCompleted_df['updatedAssets'].apply(lambda asset_changes_tuple: _asset_changes_to_dict(asset_changes_tuple)))
    return RebalanceCompleted_df

RebalanceCompleted_df = _handle_rebalance_completed(RebalanceCompleted_df)

def _handle_rebalance_started(RebalanceStarted_df: pd.DataFrame):
    def _rebalance_params_to_dict(rebalance_params_tuple):
        return {
            "destinationIn": rebalance_params_tuple[0],
            "tokenIn": rebalance_params_tuple[1],
            "amountIn": rebalance_params_tuple[2],
            "destinationOut": rebalance_params_tuple[3],
            "tokenOut": rebalance_params_tuple[4],
            "amountOut": rebalance_params_tuple[5]
        }
    
    RebalanceStarted_df[['destinationIn', 'tokenIn', 'amountIn', 'destinationOut', 'tokenOut', 'amountOut']] = pd.DataFrame.from_records(
        RebalanceStarted_df['rebalanceParams'].apply(lambda rebalance_params_tuple: _rebalance_params_to_dict(rebalance_params_tuple))
    )
    return RebalanceStarted_df

RebalanceStarted_df = _handle_rebalance_started(events['RebalanceStarted'])
RebalanceStarted_df

,event,block,transaction_index,log_index,hash,receiver,rebalanceParams,destinationIn,tokenIn,amountIn,destinationOut,tokenOut,amountOut
0,RebalanceStarted,22190481,19,51,0x67618958fa10c91e18218c3c3f7eb57509e70445e898...,0xD02b50CFc6c2903bF13638B28D081ad11515B6f9,"(0xa7569A44f348d3D70d8ad5889e50F78E33d80D35, 0...",0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48,99888696,0xb6E89d23d1E33537D383622B1955EbDa135c0b5A,0x9D39A5DE30e57443BfF2A8307A4256c8797A3497,85899837631514172722
1,RebalanceStarted,22191759,57,229,0x5513250273bfdbf939553c9ed98ce48b8479bc739634...,0xD02b50CFc6c2903bF13638B28D081ad11515B6f9,"(0xa7569A44f348d3D70d8ad5889e50F78E33d80D35, 0...",0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48,99914672,0x6F628dcCD275feA4277722D177265741031E09d7,0x390f3595bCa2Df7d23783dFd126427CCeb997BF4,98326351235806151577


In [4]:
RebalanceStarted_df[['destinationOut', 'destinationIn',  'tokenOut', 'tokenIn', 'amountOut', 'amountIn', 'hash']]

,destinationOut,destinationIn,tokenOut,tokenIn,amountOut,amountIn,hash
0,0xb6E89d23d1E33537D383622B1955EbDa135c0b5A,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,0x9D39A5DE30e57443BfF2A8307A4256c8797A3497,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48,85899837631514172722,99888696,0x67618958fa10c91e18218c3c3f7eb57509e70445e898...
1,0x6F628dcCD275feA4277722D177265741031E09d7,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,0x390f3595bCa2Df7d23783dFd126427CCeb997BF4,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48,98326351235806151577,99914672,0x5513250273bfdbf939553c9ed98ce48b8479bc739634...


In [5]:
101813602602 - 101913611260

-100008658

In [6]:
RebalanceCompleted_df

,event,block,transaction_index,log_index,hash,updatedAssets,startingIdle,startingDebt,startingTotalSupply,newIdle,newDebt,endingTotalSupply
0,RebalanceCompleted,22190481,19,78,0x67618958fa10c91e18218c3c3f7eb57509e70445e898...,"(101813602602, 200077617, 10200938913685251348...",101813602602,200077617,102009389136852513480633,101913611260,99987771,102009385384307999686043
1,RebalanceCompleted,22191759,57,269,0x5513250273bfdbf939553c9ed98ce48b8479bc739634...,"(101913611260, 99986493, 102009385384307999686...",101913611260,99986493,102009385384307999686043,102013565914,0,102009385384307999686043


In [7]:
# i think what we really need is 

In [8]:
99888696/1e6

99.888696

In [9]:
38886 + 99978657

100017543

In [10]:
# amountIn does not seem to corrospond to the right value

# here amountIn is less 99.888696 than what the vault actually got

1st is exit sUSDe -> idle
2nd is exit crvUSD/USDT idle

In [11]:
, 'RebalanceStarted_df = events['RebalanceStarted']
RebalanceStarted_df

,event,block,transaction_index,log_index,hash,receiver,rebalanceParams,destinationIn,tokenIn,amountIn,destinationOut,tokenOut,amountOut
0,RebalanceStarted,22190481,19,51,0x67618958fa10c91e18218c3c3f7eb57509e70445e898...,0xD02b50CFc6c2903bF13638B28D081ad11515B6f9,"(0xa7569A44f348d3D70d8ad5889e50F78E33d80D35, 0...",0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48,99888696,0xb6E89d23d1E33537D383622B1955EbDa135c0b5A,0x9D39A5DE30e57443BfF2A8307A4256c8797A3497,85899837631514172722
1,RebalanceStarted,22191759,57,229,0x5513250273bfdbf939553c9ed98ce48b8479bc739634...,0xD02b50CFc6c2903bF13638B28D081ad11515B6f9,"(0xa7569A44f348d3D70d8ad5889e50F78E33d80D35, 0...",0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48,99914672,0x6F628dcCD275feA4277722D177265741031E09d7,0x390f3595bCa2Df7d23783dFd126427CCeb997BF4,98326351235806151577


In [12]:
events['RebalanceStarted']['rebalanceParams'].values

array([('0xa7569A44f348d3D70d8ad5889e50F78E33d80D35', '0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48', 99888696, '0xb6E89d23d1E33537D383622B1955EbDa135c0b5A', '0x9D39A5DE30e57443BfF2A8307A4256c8797A3497', 85899837631514172722),
       ('0xa7569A44f348d3D70d8ad5889e50F78E33d80D35', '0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48', 99914672, '0x6F628dcCD275feA4277722D177265741031E09d7', '0x390f3595bCa2Df7d23783dFd126427CCeb997BF4', 98326351235806151577)],
      dtype=object)

In [13]:
RebalanceStarted_df

,event,block,transaction_index,log_index,hash,receiver,rebalanceParams,destinationIn,tokenIn,amountIn,destinationOut,tokenOut,amountOut
0,RebalanceStarted,22190481,19,51,0x67618958fa10c91e18218c3c3f7eb57509e70445e898...,0xD02b50CFc6c2903bF13638B28D081ad11515B6f9,"(0xa7569A44f348d3D70d8ad5889e50F78E33d80D35, 0...",0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48,99888696,0xb6E89d23d1E33537D383622B1955EbDa135c0b5A,0x9D39A5DE30e57443BfF2A8307A4256c8797A3497,85899837631514172722
1,RebalanceStarted,22191759,57,229,0x5513250273bfdbf939553c9ed98ce48b8479bc739634...,0xD02b50CFc6c2903bF13638B28D081ad11515B6f9,"(0xa7569A44f348d3D70d8ad5889e50F78E33d80D35, 0...",0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48,99914672,0x6F628dcCD275feA4277722D177265741031E09d7,0x390f3595bCa2Df7d23783dFd126427CCeb997BF4,98326351235806151577


In [14]:
# /// @param destinationIn The address / lp token of the destination vault that will increase
# /// @param tokenIn The address of the underlyer token that will be provided by the swapper
# /// @param amountIn The amount of the underlying LP tokens that will be received
# /// @param destinationOut The address of the destination vault that will decrease
# /// @param tokenOut The address of the underlyer token that will be received by the swapper
# /// @param amountOut The amount of the tokenOut that will be received by the swapper
# struct RebalanceParams {
#     address destinationIn;
#     address tokenIn;
#     uint256 amountIn;
#     address destinationOut;
#     address tokenOut;
#     uint256 amountOut;
# }

# this is a rebalance into Idle



# array([('0xa7569A44f348d3D70d8ad5889e50F78E33d80D35', '0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48', 99888696,
#  '0xb6E89d23d1E33537D383622B1955EbDa135c0b5A', '0x9D39A5DE30e57443BfF2A8307A4256c8797A3497', 85899837631514172722)],
